# Stacking and Blending

There are three ways to combine models, and this section has covered two of them:

| family | how it combines | example |
|---|---|---|
| **bagging** | average many models trained on resampled data | [[random-forest]] |
| **boosting** | fit each model to what the previous ones got wrong | [[gradient-boosting-from-scratch]] |
| **stacking** | train a model to combine the *predictions* of other models | this page |

Bagging reduces variance, boosting reduces bias, and stacking does something
different from both: it **learns how much to trust each model**, and it can
learn that the answer varies.

The whole method rests on one requirement, and getting it wrong silently ruins
everything: the combiner must be trained on predictions the base models made for
rows they **did not see during training**. That means out-of-fold predictions,
which is why this page sits next to [[cross-validation]] and [[data-leakage]].

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

from sklearn.datasets import make_classification
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     cross_val_predict, train_test_split)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss

banner("FOUR BASE MODELS THAT DISAGREE")

# A deliberately HETEROGENEOUS target: part of the signal is linear in the first
# three features, and part is a pure interaction (the sign of a product), which
# no linear model can represent at all. Different model families will therefore
# succeed on different halves of the problem -- which is the situation stacking
# exists for.
rng = np.random.RandomState(0)
n_rows = 2000
X = rng.randn(n_rows, 6)
linear_part = X[:, :3] @ np.array([1.6, -1.3, 1.0])
interaction  = 3.0 * np.sign(X[:, 3] * X[:, 4])
y = ((linear_part + interaction + rng.randn(n_rows) * 0.7) > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.35, random_state=0,
                                          stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=1)

print(f"  {n_rows} rows, 6 features, class balance {np.bincount(y).tolist()}")
print(f"  signal = a linear term in features 0-2, PLUS sign(x3 * x4)")
print(f"  the second term is invisible to any linear model")

bases = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    'forest'  : RandomForestClassifier(n_estimators=300, random_state=0),
    # 1-NN is included deliberately. It memorises its training set perfectly
    # and generalises only moderately, which is exactly the profile that makes
    # in-sample meta-features dangerous -- see section 1.
    'nn1'     : make_pipeline(StandardScaler(), KNeighborsClassifier(1)),
    'nbayes'  : GaussianNB(),
}

step("Each on its own")
print(f"  {'model':>10}  {'CV accuracy':>12}")
print("  " + "-" * 26)
for nm, m in bases.items():
    print(f"  {nm:>10}  {cross_val_score(m, X_tr, y_tr, cv=cv).mean():>12.4f}")
print("  " + "-" * 26)

step("Do they make the SAME mistakes?")
preds = {}
for nm, m in bases.items():
    preds[nm] = cross_val_predict(m, X_tr, y_tr, cv=cv)
names = list(bases)
print(f"  pairwise agreement on predicted label:")
print(f"  {'':>10}" + "".join(f"{n:>10}" for n in names))
for a in names:
    row = "".join(f"{np.mean(preds[a]==preds[b]):>10.3f}" for b in names)
    print(f"  {a:>10}{row}")
print()
wrong = {nm: preds[nm] != y_tr for nm in names}
all_wrong = np.all([wrong[n] for n in names], axis=0)
any_wrong = np.any([wrong[n] for n in names], axis=0)
print(f"  rows at least one model gets wrong : {any_wrong.sum():>4}"
      f"  ({any_wrong.mean():.1%})")
print(f"  rows ALL FOUR get wrong            : {all_wrong.sum():>4}"
      f"  ({all_wrong.mean():.1%})")
print()
print(f"  The gap between those two numbers is the entire opportunity. On")
print(f"  {any_wrong.sum()-all_wrong.sum()} rows, somebody in the ensemble already knew the answer.")
print(f"  Combining cannot help on the {all_wrong.sum()} rows nobody gets right.")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
  FOUR BASE MODELS THAT DISAGREE
  2000 rows, 6 features, class balance [980, 1020]
  signal = a linear term in features 0-2, PLUS sign(x3 * x4)
  the second term is invisible to any linear model

Each on its own
       model   CV accuracy
  --------------------------
    logistic        0.6046
      forest        0.9046
         nn1        0.7646
      nbayes        0.6015
  --------------------------

Do they make the SAME mistakes?
  pairwise agreement on predicted label:
              logistic    forest       nn1    nbayes
    logistic     1.000     0.651     0.588     0.952
      forest     0.651     1.000     0.771     0.649
         nn1     0.588     0.771     1.000     0.589
      nbayes     0.952     0.649     0.589     1.000

  rows at least one model gets wrong :  713  (54.8%)
  rows ALL FOUR get wrong            :   45  (3.5%)

  The gap between those two numbers is the entire op

## 1. The out-of-fold construction

The combiner (the **meta-learner**) needs a training set whose features are the
base models' predictions. The question is where those predictions come from.

**The wrong way**, and it is tempting: fit each base model on the training set,
predict on that same training set, and train the meta-learner on the result. The
base models have already seen those rows. An overfitted model looks *perfect* on
its own training data, so the meta-learner concludes it should trust the
overfitted model completely — and that model is the one that generalises worst.

**The right way** — out-of-fold predictions:

1. Split the training data into $k$ folds.
2. For each fold $i$: train each base model on the other $k-1$ folds, predict fold $i$.
3. Every row now has a prediction from a model that never saw it. Stack those columns into a meta-feature matrix $Z$.
4. Train the meta-learner on $(Z, y)$.
5. Refit each base model on **all** the training data, for use at prediction time.

The cost is $k+1$ fits per base model. The demonstration below shows why paying
it is not optional.

In [2]:
banner("OUT-OF-FOLD PREDICTIONS, AND WHAT HAPPENS WITHOUT THEM")

step("Build Z the right way: cross_val_predict gives out-of-fold probabilities")
Z_oof = np.column_stack([
    cross_val_predict(m, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
    for m in bases.values()])
print(f"  Z shape {Z_oof.shape}  -- one column per base model, one row per"
      f" training row")
print(f"  {'row':>5}  " + "".join(f"{n:>10}" for n in names) + f"{'y':>5}")
for i in range(6):
    print(f"  {i:>5}  " + "".join(f"{Z_oof[i,j]:>10.4f}"
                                  for j in range(len(names))) + f"{y_tr[i]:>5}")

step("Build Z the WRONG way: in-sample predictions")
Z_bad = np.column_stack([
    m.fit(X_tr, y_tr).predict_proba(X_tr)[:, 1] for m in bases.values()])
# test-set meta-features, needed by the comparisons below
Z_te = np.column_stack([
    m.fit(X_tr, y_tr).predict_proba(X_te)[:, 1] for m in bases.values()])
print(f"  How well does each column predict y, in-sample vs out-of-fold?")
print(f"  {'model':>10}  {'in-sample acc':>14}  {'out-of-fold acc':>16}")
print("  " + "-" * 44)
for j, nm in enumerate(names):
    print(f"  {nm:>10}  {accuracy_score(y_tr, Z_bad[:,j]>0.5):>14.4f}"
          f"  {accuracy_score(y_tr, Z_oof[:,j]>0.5):>16.4f}")
print("  " + "-" * 44)
print(f"  Two models score 1.0000 in-sample. Neither is perfect -- both have")
print(f"  memorised the training rows. Look at what happens to 'nn1' when it")
print(f"  is asked about rows it has NOT seen: {accuracy_score(y_tr, Z_oof[:,names.index('nn1')]>0.5):.4f}.")
print(f"  A meta-learner shown the left column has no way to know that.")

step("What the meta-learner learns from each version")
meta_bad = LogisticRegression(max_iter=2000).fit(Z_bad, y_tr)
meta_oof = LogisticRegression(max_iter=2000).fit(Z_oof, y_tr)
print(f"  {'model':>10}  {'weight (leaky Z)':>17}  {'weight (honest Z)':>18}")
print("  " + "-" * 50)
for j, nm in enumerate(names):
    print(f"  {nm:>10}  {meta_bad.coef_[0][j]:>17.4f}"
          f"  {meta_oof.coef_[0][j]:>18.4f}")
print("  " + "-" * 50)
dom_bad = names[int(np.argmax(meta_bad.coef_[0]))]
dom_oof = names[int(np.argmax(meta_oof.coef_[0]))]
print(f"  leaky Z  puts the most weight on '{dom_bad}'")
print(f"  honest Z puts the most weight on '{dom_oof}'")
print()
print(f"  '{dom_bad}' has {accuracy_score(y_tr, Z_bad[:,names.index(dom_bad)]>0.5):.0%} in-sample accuracy and"
      f" {accuracy_score(y_te, Z_te[:,names.index(dom_bad)]>0.5):.1%} on the test set.")
print(f"  '{dom_oof}' has {accuracy_score(y_te, Z_te[:,names.index(dom_oof)]>0.5):.1%} on the test set.")
print(f"  The leak did not add noise -- it pointed the meta-learner at the")
print(f"  wrong model, confidently.")

step("The cost, measured on the held-out test set")
acc_bad = accuracy_score(y_te, meta_bad.predict(Z_te))
acc_oof = accuracy_score(y_te, meta_oof.predict(Z_te))
base_te = {nm: accuracy_score(y_te, m.fit(X_tr, y_tr).predict(X_te))
           for nm, m in bases.items()}
best_base = max(base_te.values())
best_name = max(base_te, key=base_te.get)
print(f"  {'base model':>22}  {'test accuracy':>14}")
print("  " + "-" * 40)
for nm, a in sorted(base_te.items(), key=lambda kv: -kv[1]):
    print(f"  {nm:>22}  {a:>14.4f}")
print("  " + "-" * 40)
print(f"  {'stacking, leaky Z':>22}  {acc_bad:>14.4f}")
print(f"  {'stacking, out-of-fold':>22}  {acc_oof:>14.4f}")
print("  " + "-" * 40)
print()
print(f"  out-of-fold vs leaky Z            : {acc_oof-acc_bad:+.4f}")
print(f"  out-of-fold vs best single model  : {acc_oof-best_base:+.4f}"
      f"   (best was '{best_name}')")

  OUT-OF-FOLD PREDICTIONS, AND WHAT HAPPENS WITHOUT THEM

Build Z the right way: cross_val_predict gives out-of-fold probabilities
  Z shape (1300, 4)  -- one column per base model, one row per training row
    row    logistic    forest       nn1    nbayes    y
      0      0.2475    0.5667    0.0000    0.3190    0
      1      0.7201    0.8833    1.0000    0.7237    1
      2      0.3590    0.5000    1.0000    0.3551    1
      3      0.3771    0.6900    1.0000    0.3797    1
      4      0.6261    0.2700    0.0000    0.6333    0
      5      0.3620    0.2067    0.0000    0.3890    0

Build Z the WRONG way: in-sample predictions
  How well does each column predict y, in-sample vs out-of-fold?
       model   in-sample acc   out-of-fold acc
  --------------------------------------------
    logistic          0.6092            0.6046
      forest          1.0000            0.9046
         nn1          1.0000            0.7646
      nbayes          0.6085            0.6015
  -------------

## 2. What the meta-learner actually learns

With a logistic meta-learner the fitted coefficients are readable: they are how
much each base model is trusted, on the log-odds scale. Two things are worth
noticing.

**The weights are not proportional to accuracy.** A model that is slightly worse
overall but wrong in *different places* can receive more weight than a slightly
better one that duplicates an existing member. Stacking rewards
complementarity, not just quality — which is why an ensemble of four mediocre,
diverse models often beats four good, similar ones.

**Weights can be negative.** A model that is reliably wrong in a particular
region is informative; the meta-learner can subtract it. This is something no
simple average can do.

In [3]:
banner("WEIGHTS, DIVERSITY, AND THE SIMPLE-AVERAGE BASELINE")

step("Fitted trust vs standalone accuracy")
print(f"  {'model':>10}  {'own CV acc':>11}  {'meta weight':>12}"
      f"  {'corr with others':>17}")
print("  " + "-" * 56)
for j, nm in enumerate(names):
    own = cross_val_score(bases[nm], X_tr, y_tr, cv=cv).mean()
    oth = [k for k in range(len(names)) if k != j]
    corr = np.mean([np.corrcoef(Z_oof[:, j], Z_oof[:, k])[0, 1] for k in oth])
    print(f"  {nm:>10}  {own:>11.4f}  {meta_oof.coef_[0][j]:>12.4f}"
          f"  {corr:>17.4f}")
print("  " + "-" * 56)
neg = [names[j] for j in range(len(names)) if meta_oof.coef_[0][j] < 0]
print(f"  Note the NEGATIVE weights: {', '.join(neg) if neg else '(none here)'}.")
print(f"  A simple average cannot do that. A model that is reliably wrong in")
print(f"  a particular region is informative, and the meta-learner subtracts")
print(f"  it rather than being dragged along by it.")

step("Stacking vs simple averaging")
avg_te = Z_te.mean(1)
acc_avg = accuracy_score(y_te, avg_te > 0.5)
print(f"  {'method':>26}  {'test accuracy':>14}  {'test log loss':>14}")
print("  " + "-" * 58)
print(f"  {'best single model':>26}  {best_base:>14.4f}  {'':>14}")
print(f"  {'simple average of the 4':>26}  {acc_avg:>14.4f}"
      f"  {log_loss(y_te, np.clip(avg_te,1e-9,1-1e-9)):>14.4f}")
print(f"  {'stacked (logistic meta)':>26}  {acc_oof:>14.4f}"
      f"  {log_loss(y_te, meta_oof.predict_proba(Z_te)[:,1]):>14.4f}")
print("  " + "-" * 58)
print()
print(f"  Look at the middle row. The simple average scores {acc_avg:.4f} --"
      f" WORSE than")
print(f"  simply using '{best_name}' alone ({best_base:.4f}). Two of the four base")
print(f"  models are near-useless on this problem, and averaging gives them")
print(f"  the same vote as the good ones.")
print()
print(f"  Stacking's reliable win is exactly there: over the average,"
      f" {acc_oof-acc_avg:+.4f}.")
print(f"  Its win over the single best model is much smaller"
      f" ({acc_oof-best_base:+.4f}), and")
print(f"  that is the honest general picture -- stacking usually buys a")
print(f"  percent or two over your best model, which matters in a competition")
print(f"  and rarely justifies four models in production.")

step("Check against scikit-learn's StackingClassifier")
sk = StackingClassifier(
    estimators=[(n, bases[n]) for n in names],
    final_estimator=LogisticRegression(max_iter=2000),
    cv=cv, stack_method='predict_proba').fit(X_tr, y_tr)
sk_acc = accuracy_score(y_te, sk.predict(X_te))
print(f"  by hand      : {acc_oof:.4f}")
print(f"  scikit-learn : {sk_acc:.4f}")
print(f"  difference   : {abs(acc_oof-sk_acc):.4f}")
print()
print(f"  Small differences are expected: StackingClassifier passes all 2")
print(f"  probability columns per estimator by default and refits the bases")
print(f"  on the full training set slightly differently. The construction is")
print(f"  the same one built above.")

  WEIGHTS, DIVERSITY, AND THE SIMPLE-AVERAGE BASELINE

Fitted trust vs standalone accuracy
       model   own CV acc   meta weight   corr with others
  --------------------------------------------------------
    logistic       0.6046       -0.4883             0.6061
      forest       0.9046       10.1849             0.5770
         nn1       0.7646        0.6994             0.3861
      nbayes       0.6015       -1.2488             0.6086
  --------------------------------------------------------
  Note the NEGATIVE weights: logistic, nbayes.
  A simple average cannot do that. A model that is reliably wrong in
  a particular region is informative, and the meta-learner subtracts
  it rather than being dragged along by it.

Stacking vs simple averaging
                      method   test accuracy   test log loss
  ----------------------------------------------------------
           best single model          0.9000                
     simple average of the 4          0.8043          

## 3. Blending, and the failure mode of both

**Blending** is the cheap variant: instead of $k$-fold out-of-fold predictions,
hold out a single validation slice (say 20%), train the bases on the rest,
and train the meta-learner on that one slice.

- cheaper — one fit per base model instead of $k+1$
- simpler to get right — there is no fold bookkeeping to botch
- but the meta-learner sees far less data, and the base models never train on the holdout

Use blending when base models are expensive; stacking when data is scarce.

**The failure mode both share** is the one from [[cross-validation]]: the
meta-learner is itself a model selected using the training folds. Adding base
models until the stacked CV score peaks is a search, and the winner's score is
biased upward for exactly the reasons in that page. A stacked ensemble tuned this
way needs an outer loop to be reported honestly — and stacking makes the problem
worse than usual, because the number of things being compared is large and the
gains being chased are small.

In [4]:
banner("BLENDING, AND WHY A STACKED SCORE NEEDS AN OUTER LOOP")

step("Blending: one holdout slice instead of k folds")
X_base, X_blend, y_base, y_blend = train_test_split(
    X_tr, y_tr, test_size=0.25, random_state=0, stratify=y_tr)
Zb = np.column_stack([m.fit(X_base, y_base).predict_proba(X_blend)[:, 1]
                      for m in bases.values()])
meta_bl = LogisticRegression(max_iter=2000).fit(Zb, y_blend)
Zt = np.column_stack([m.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
                      for m in bases.values()])
print(f"  meta-learner trained on {len(y_blend)} rows (blending)"
      f" vs {len(y_tr)} (stacking)")
print(f"  base model fits: {len(bases)} (blending)"
      f" vs {len(bases)*(cv.get_n_splits()+1)} (stacking)")
print()
print(f"  blended  test accuracy : {accuracy_score(y_te, meta_bl.predict(Zt)):.4f}")
print(f"  stacked  test accuracy : {acc_oof:.4f}")

step("Selection bias: adding members until the score peaks")
from itertools import combinations
print(f"  Trying every non-empty subset of the {len(names)} base models.")
print(f"  The out-of-fold columns Z are computed once and reused, so each")
print(f"  subset only needs the meta-learner refitted -- the base models do")
print(f"  not change with the subset.")
print()
print(f"  {'subset':>34}  {'CV score':>9}  {'test':>7}")
print("  " + "-" * 56)
results = []
for k in range(1, len(names)+1):
    for combo in combinations(names, k):
        cols = [names.index(nm) for nm in combo]
        cvs = cross_val_score(LogisticRegression(max_iter=2000),
                              Z_oof[:, cols], y_tr, cv=cv).mean()
        meta_c = LogisticRegression(max_iter=2000).fit(Z_oof[:, cols], y_tr)
        te = accuracy_score(y_te, meta_c.predict(Z_te[:, cols]))
        results.append((combo, cvs, te))
results.sort(key=lambda r: -r[1])
for combo, cvs, te in results[:5]:
    print(f"  {'+'.join(combo):>34}  {cvs:>9.4f}  {te:>7.4f}")
print("  " + "-" * 56)
best_cv = results[0]
best_test = max(results, key=lambda r: r[2])
print(f"  chosen by CV : {'+'.join(best_cv[0])}  (CV {best_cv[1]:.4f},"
      f" test {best_cv[2]:.4f})")
print(f"  best on test : {'+'.join(best_test[0])}  (CV {best_test[1]:.4f},"
      f" test {best_test[2]:.4f})")
print(f"  same subset? {best_cv[0] == best_test[0]}")
print()
print(f"  cost of choosing by CV instead of an oracle:"
      f" {best_test[2]-best_cv[2]:+.4f} test accuracy")
print()
print(f"  {len(results)} subsets were compared, so the winning CV score is the")
print(f"  maximum of {len(results)} noisy numbers. Two consequences, and the")
print(f"  second is the one people miss:")
print(f"    1. that maximum is biased upward as an estimate of performance")
print(f"    2. the ARGMAX is unreliable too -- the subset CV picks is often")
print(f"       not the one that does best on new data, as above.")
print()
print(f"  Both are the same phenomenon from [[cross-validation]]: with"
      f" {len(results)} candidates")
print(f"  and differences this small, the ranking is mostly noise. Report a")
print(f"  stacked ensemble from an outer loop, or do not report a number.")

  BLENDING, AND WHY A STACKED SCORE NEEDS AN OUTER LOOP

Blending: one holdout slice instead of k folds
  meta-learner trained on 325 rows (blending) vs 1300 (stacking)
  base model fits: 4 (blending) vs 24 (stacking)

  blended  test accuracy : 0.8900
  stacked  test accuracy : 0.9043

Selection bias: adding members until the score peaks
  Trying every non-empty subset of the 4 base models.
  The out-of-fold columns Z are computed once and reused, so each
  subset only needs the meta-learner refitted -- the base models do
  not change with the subset.

                              subset   CV score     test
  --------------------------------------------------------
              logistic+forest+nbayes     0.9115   0.9100
                       forest+nbayes     0.9108   0.9100
                     logistic+forest     0.9092   0.9114
                   forest+nn1+nbayes     0.9054   0.9029
                              forest     0.9046   0.9000
  -------------------------------------

## 4. What to take away

**Stacking learns trust; averaging assumes it.** The meta-learner discovers how
much each base model is worth, and can assign negative weight to one that is
usefully wrong. That flexibility is what it buys over a simple average — and a
simple average is a genuinely strong baseline that costs nothing, so make the
comparison before adding the complexity.

**Out-of-fold predictions are the whole method.** Train the meta-learner on
in-sample predictions and it will learn to trust whichever base model memorised
best. This is [[data-leakage]] with extra steps, and it is invisible unless you
check.

**Diversity beats individual quality.** The opportunity is the gap between "at
least one model is wrong" and "all models are wrong". Adding a fifth model that
agrees with an existing one widens neither.

**And the stacked score needs an outer loop.** Choosing which models to include,
by comparing cross-validated stacked scores, is a search — with the same upward
bias as any other search.

### Verified in this notebook

- Base models agree on most rows but the "at least one wrong" and "all wrong" counts differ substantially, which is the room stacking has to work in
- In-sample meta-features give 1-NN — which memorises perfectly and generalises at 77% — the *highest* meta weight, and the ensemble collapses to exactly 1-NN's test accuracy. Out-of-fold meta-features pick the forest instead, worth **+0.13** test accuracy
- The meta-learner assigns negative weights, which a simple average cannot
- The simple average scores *below* the best single model, because two of the four bases are near-useless and averaging gives them equal votes — this is the gap stacking reliably closes
- Hand-built stacking lands close to `StackingClassifier`, with the remaining difference explained rather than hidden
- All 15 subsets compared: the CV-selected subset is not the test-best one

### Where this connects

[[cross-validation]] builds the out-of-fold machinery this depends on.
[[data-leakage]] is what in-sample meta-features are an instance of.
[[random-forest]] and [[gradient-boosting-from-scratch]] are the other two ways
to combine models. [[probability-calibration]] matters here too — a
badly-calibrated base model contributes a distorted meta-feature.